# Cross-model replication — Qwen and Mistral
Run cells in order. Upload original Gemma raw CSV. Never share your API key. Pilots already completed separately. No official requests until validation succeeds.

In [ ]:
import csv, json, hashlib, time, requests, io, os
from pathlib import Path
from getpass import getpass
from google.colab import files
uploaded = files.upload()
source_name = next((name for name in uploaded if name.lower().endswith('.csv') and 'gemma_tutor_raw' in name), None)
assert source_name, 'Upload the original gemma_tutor_raw CSV.'
original_bytes = uploaded[source_name]
source_hash = hashlib.sha256(original_bytes).hexdigest()
rows = list(csv.DictReader(io.StringIO(original_bytes.decode('utf-8-sig'))))
print('Original SHA256:', source_hash)
print('Rows:', len(rows))

In [ ]:
from collections import Counter
keys = [(r['history_id'],r['condition'],str(r['repetition'])) for r in rows]
assert len(rows)==60 and len(set(keys))==60
assert set(r['history_id'] for r in rows)=={f'H{i:02}' for i in range(1,11)}
assert set(r['condition'] for r in rows)=={'CN','CU','IN'}
assert all(v==2 for v in Counter((r['history_id'],r['condition']) for r in rows).values())
assert all(not r['error'].strip() for r in rows)
assert all(float(r['temperature'])==0.7 and int(r['max_tokens'])==250 for r in rows)
for r in rows:
    m=json.loads(r['prompt_messages_json'])
    assert m[0]['role']=='system' and m[-1]['role']=='user'
    assert isinstance(m,list) and len(m)>1
assert len(set(r['prompt_messages_json'] for r in rows))==30
print('PASS: 60 unique trials, 30 exact prompts, balanced histories, no errors, fixed parameters')

## Configuration
Provider is pinned by name, fallback disabled, strict parameter support. OpenRouter may route among multiple endpoints at the same provider; preserve response IDs and usage. Qwen requests reasoning disabled, matching the successful pilot.

In [ ]:
api_key = getpass('OpenRouter API key (hidden): ')
url = 'https://openrouter.ai/api/v1/chat/completions'
headers={'Authorization':f'Bearer {api_key}', 'Content-Type':'application/json'}
configs=[
 {'model':'qwen/qwen3.5-27b','provider':'Alibaba','reasoning':{'enabled':False}},
 {'model':'mistralai/mistral-small-3.2-24b-instruct','provider':'DeepInfra'}
]
print('Models configured:', [c['model'] for c in configs])

## Official 120 trials
This cell makes paid requests. Results are checkpointed after each request, with no automatic retry. If interrupted, upload the checkpoint file in a fresh session before resuming; do not simply restart from scratch. Do not run this cell twice without restoring the saved checkpoint.

In [ ]:
out_path=Path('/content/cross_model_raw_checkpoint.csv')
columns=['history_id','condition','repetition','timestamp_utc','model_requested','model_reported','provider_requested','temperature','max_tokens','prompt_messages_json','raw_response','error','usage_json','response_id','finish_reason','reasoning_text','reasoning_details_json','source_sha256']
if out_path.exists():
    with out_path.open(encoding='utf-8',newline='') as f: results=list(csv.DictReader(f))
else: results=[]
completed={(r['model_requested'],r['history_id'],r['condition'],str(r['repetition'])) for r in results}
print('Already recorded:',len(completed))
from datetime import datetime,timezone
for config in configs:
  for r in rows:
    key=(config['model'],r['history_id'],r['condition'],str(r['repetition']))
    if key in completed: continue
    payload={
      'model':config['model'],
      'provider':{'order':[config['provider']],'allow_fallbacks':False,'require_parameters':True},
      'messages':json.loads(r['prompt_messages_json']),
      'temperature':0.7,'max_tokens':250,
    }
    if 'reasoning' in config: payload['reasoning']=config['reasoning']
    record={k:'' for k in columns}
    record.update({k:r[k] for k in ('history_id','condition','repetition','temperature','max_tokens','prompt_messages_json')})
    record.update(model_requested=config['model'],provider_requested=config['provider'],source_sha256=source_hash,timestamp_utc=datetime.now(timezone.utc).isoformat())
    try:
      resp=requests.post(url,headers=headers,json=payload,timeout=90)
      if resp.status_code!=200: raise RuntimeError(f'HTTP {resp.status_code}: {resp.text[:500]}')
      data=resp.json(); choice=data['choices'][0]; message=choice['message']
      record.update(model_reported=data.get('model',''),raw_response=message.get('content') or '',usage_json=json.dumps(data.get('usage',{})),response_id=data.get('id',''),finish_reason=choice.get('finish_reason',''),reasoning_text=message.get('reasoning') or '',reasoning_details_json=json.dumps(message.get('reasoning_details')))
    except Exception as exc:
      record['error']=str(exc)
    results.append(record);completed.add(key)
    with out_path.open('w',encoding='utf-8',newline='') as f:
      w=csv.DictWriter(f,fieldnames=columns);w.writeheader();w.writerows(results)
    print(f"{len(results):3}/120 {config['model']} {r['history_id']} {r['condition']} rep{r['repetition']} {'ERROR: '+record['error'] if record['error'] else record['finish_reason']}")
    if record['error']:
      print('Stopped on first error. Download checkpoint and ask for help before continuing.')
      raise RuntimeError(record['error'])
    time.sleep(0.4)
print('Official collection complete')

In [ ]:
assert len(results)==120, f'Expected 120 rows; found {len(results)}'
assert all(not r['error'] for r in results)
assert all(r['raw_response'].strip() for r in results)
assert all(r['finish_reason']=='stop' for r in results), 'Inspect non-stop finish reasons before coding'
assert all(r['source_sha256']==source_hash for r in results)
original_lookup={(r['history_id'],r['condition'],str(r['repetition'])):r['prompt_messages_json'] for r in rows}
assert all(r['prompt_messages_json']==original_lookup[(r['history_id'],r['condition'],str(r['repetition']))] for r in results)
assert Counter(r['model_requested'] for r in results)=={c['model']:60 for c in configs}
print('PASS: 120 successful responses, exact original prompts, 60 per model')
print('Finish reasons:',Counter(r['finish_reason'] for r in results))
print('Reported models:',Counter(r['model_reported'] for r in results))
print('Reasoning tokens:',Counter((r['model_requested'],json.loads(r['usage_json']).get('completion_tokens_details',{}).get('reasoning_tokens')) for r in results))
files.download(str(out_path))